# Bài tập chương 2

In [ ]:

import sqlite3
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

conn = sqlite3.connect(':memory:')

setup_sql = '''
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS course;

CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);

INSERT INTO student(student_id, name, class, course_id, score) VALUES
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan',      'Kinh Te',  34, 9.2),
(3, 'Pham Van Nam',      'Toan Tin', NULL, 7.9),
(4, 'Le Thanh Huyen',    'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh',       'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh',    'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung',     'Kinh Te',  34, 9.2),
(8, 'Ho Ngoc Mai',       'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc',    'Kinh Te',  NULL, 7.2),
(10,'Cao Thi Hanh',      'May Tinh', NULL, 7.0);

INSERT INTO course(id, course_name) VALUES
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc');
'''
conn.executescript(setup_sql)

def q(sql):
    return pd.read_sql_query(sql, conn)

def show(title, sql):
    display(Markdown(f"### {title}"))
    print(sql.strip())
    display(q(sql))

display(Markdown("## Dữ liệu ban đầu"))
display(Markdown("### Bảng `student`"))
display(q("SELECT * FROM student ORDER BY student_id"))
display(Markdown("### Bảng `course`"))
display(q("SELECT * FROM course ORDER BY id"))


## Dữ liệu ban đầu

### Bảng `student`

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


### Bảng `course`

,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


## 1.Kết nối hai bảng

In [ ]:
show("Tích Descartes giữa student và course (dùng dấu phẩy)", '''
SELECT s.student_id, s.name, s.class, s.course_id AS student_course_id, s.score,
       c.id AS course_id_from_course, c.course_name
FROM student AS s, course AS c
ORDER BY s.student_id, c.id;
''')

inner_sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
INNER JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id;
'''

left_sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id;
'''

right_sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY c.id, s.student_id;
'''
full_sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY student_id, id;
'''

for title, sql in [
    ("INNER JOIN", inner_sql),
    ("LEFT JOIN", left_sql),
    ("RIGHT JOIN", right_sql),
    ("FULL OUTER JOIN", full_sql),
]:
    df = q(sql)
    display(Markdown(f"### {title}"))
    print(sql.strip())
    print(f"Số dòng: {len(df)}")
    display(df)

### Tích Descartes giữa student và course (dùng dấu phẩy)

SELECT s.student_id, s.name, s.class, s.course_id AS student_course_id, s.score, 
       c.id AS course_id_from_course, c.course_name 
FROM student AS s, course AS c 
ORDER BY s.student_id, c.id;


,student_id,name,class,student_course_id,score,course_id_from_course,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


### INNER JOIN

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
INNER JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id;
Số dòng: 3


...

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY c.id, s.student_id;
Số dòng: 4


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


### FULL OUTER JOIN

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY student_id, id;
Số dòng: 11


,student_id,name,class,course_id,score,id,course_name
0,NaN,None,None,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None


KẾT LUẬN PHẦN JOIN

Qua việc thực hiện các phép JOIN giữa hai bảng `student` và `course`, có thể rút ra các nhận xét sau:

 🔹 1. Tích Descartes (CROSS JOIN)
- Sử dụng cú pháp `FROM student, course` để tạo tích Descartes.
- Kết quả tạo ra **mọi tổ hợp giữa sinh viên và môn học**.
- Số dòng kết quả = số dòng bảng `student` × số dòng bảng `course`.
- Không phản ánh dữ liệu thực tế, chủ yếu dùng để minh họa hoặc theo yêu cầu đề bài.

---

 🔹 2. INNER JOIN
- Chỉ lấy các bản ghi có **liên kết hợp lệ** giữa `student.course_id` và `course.id`.
- Phản ánh dữ liệu chính xác trong thực tế.
- Trong bài, thu được các sinh viên có môn học hợp lệ.

---
 🔹 3. LEFT JOIN
- Giữ toàn bộ dữ liệu từ bảng `student`.
- Nếu không có môn học tương ứng → giá trị `NULL`.
- Dùng khi muốn không làm mất dữ liệu phía sinh viên.

---
 🔹 4. RIGHT JOIN (giả lập)
- Do SQLite không hỗ trợ `RIGHT JOIN`, sử dụng `LEFT JOIN` đảo bảng.
- Giữ toàn bộ dữ liệu từ bảng `course`.
- Dùng khi muốn không làm mất dữ liệu phía môn học.

---
 🔹 5. FULL OUTER JOIN (giả lập)
- Kết hợp `LEFT JOIN` và `RIGHT JOIN` bằng `UNION`.
- Giữ toàn bộ dữ liệu từ cả hai bảng.
- Dùng khi cần tổng hợp đầy đủ thông tin.


## 2.Cập nhật `course_id` còn thiếu, xóa bản ghi có môn học không tồn tại

In [ ]:

update_delete_sql = '''
UPDATE student
SET course_id = CASE student_id
    WHEN 3 THEN 26
    WHEN 9 THEN 34
    WHEN 10 THEN 12
END
WHERE student_id IN (3, 9, 10);

-- Xóa các bản ghi có course_id không tồn tại trong bảng course
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course);
'''

conn.executescript(update_delete_sql)

display(Markdown("### SQL đã dùng"))
print(update_delete_sql.strip())

display(Markdown("### Bảng `student` sau khi làm sạch dữ liệu"))
display(q("SELECT * FROM student ORDER BY student_id"))


### SQL đã dùng

UPDATE student
SET course_id = CASE student_id
    WHEN 3 THEN 26
    WHEN 9 THEN 34
    WHEN 10 THEN 12
END
WHERE student_id IN (3, 9, 10);

-- Xóa các bản ghi có course_id không tồn tại trong bảng course
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course);


### Bảng `student` sau khi làm sạch dữ liệu

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,34,7.2
5,10,Cao Thi Hanh,May Tinh,12,7.0


Dữ liệu bảng student đã được chuẩn hóa bằng cách cập nhật course_id hợp lệ và loại bỏ các bản ghi không tồn tại trong bảng course. Sau xử lý, toàn bộ dữ liệu đều hợp lệ và sẵn sàng cho các bước phân tích tiếp theo.

In [ ]:

show("2a. Tổng số sinh viên và điểm trung bình theo lớp", '''
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
ORDER BY class;
''')

show("2b. Tổng số sinh viên và điểm trung bình theo môn học", '''
SELECT s.course_id,
       c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY s.course_id, c.course_name
ORDER BY s.course_id;
''')

show("2c. Phân loại thi đua theo điểm trung bình từng môn học", '''
SELECT s.course_id,
       c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6.0 AND AVG(s.score) <= 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY s.course_id, c.course_name
ORDER BY s.course_id;
''')


### 2a. Tổng số sinh viên và điểm trung bình theo lớp

SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
ORDER BY class;


,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


### 2b. Tổng số sinh viên và điểm trung bình theo môn học

SELECT s.course_id,
       c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY s.course_id, c.course_name
ORDER BY s.course_id;


,course_id,course_name,total_students,avg_score
0,12,Giai tich,2,6.85
1,26,Tin hoc,1,7.90
2,34,Thong ke,3,8.53


### 2c. Phân loại thi đua theo điểm trung bình từng môn học

SELECT s.course_id,
       c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6.0 AND AVG(s.score) <= 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c
    ON s.course_id = c.id
GROUP BY s.course_id, c.course_name
ORDER BY s.course_id;


,course_id,course_name,avg_score,xep_loai
0,12,Giai tich,6.85,Tot
1,26,Tin hoc,7.90,Tot
2,34,Thong ke,8.53,Tot


(a) Theo lớp học:

Lớp Kinh Tế có số lượng sinh viên nhiều nhất (3 sinh viên) và điểm trung bình cao nhất (8.53).
Lớp Máy Tính có điểm trung bình thấp hơn (6.85).
Lớp Toán Tin có ít sinh viên nhất (1 sinh viên) với điểm trung bình 7.90.

(b) Theo môn học:

Môn Thống kê (34) có nhiều sinh viên nhất (3 sinh viên) và điểm trung bình cao nhất (8.53).
Môn Giải tích (12) có điểm trung bình thấp nhất (6.85).
Môn Tin học (26) có số lượng ít nhất (1 sinh viên), điểm trung bình 7.90.

(c) Phân loại thi đua:

Tất cả các môn học đều có điểm trung bình nằm trong khoảng từ 6.0 đến 8.9, do đó đều được xếp loại "Tốt".
Không có môn nào đạt mức "Xuất sắc" hoặc "Kém".

## 3.Xếp hạng sinh viên

In [ ]:

show("3a. Xếp hạng theo điểm số toàn bộ sinh viên", '''
SELECT student_id, name, class, course_id, score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS rank_by_score
FROM student
ORDER BY rank_by_score, student_id;
''')

show("Top 3 sinh viên điểm cao nhất (toàn bộ)", '''
WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score DESC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY rn;
''')

show("Top 3 sinh viên điểm thấp nhất (toàn bộ)", '''
WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score ASC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY rn;
''')


### 3a. Xếp hạng theo điểm số toàn bộ sinh viên

SELECT student_id, name, class, course_id, score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS rank_by_score
FROM student
ORDER BY rank_by_score, student_id;


,student_id,name,class,course_id,score,rank_by_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,2
3,9,Duong Huu Phuc,Kinh Te,34,7.2,3
4,10,Cao Thi Hanh,May Tinh,12,7.0,4
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,5


### Top 3 sinh viên điểm cao nhất (toàn bộ)

WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score DESC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY rn;


,student_id,name,class,course_id,score,rn
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,3,Pham Van Nam,Toan Tin,26,7.9,3


### Top 3 sinh viên điểm thấp nhất (toàn bộ)

WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score ASC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY rn;


,student_id,name,class,course_id,score,rn
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


Kết luận: Kết quả xếp hạng cho thấy hai sinh viên đạt điểm cao nhất là Trần Thị Lan và Bùi Tiến Dũng (9.2), trong khi Nguyễn Minh Hoàng có điểm thấp nhất (6.7). Nhìn chung, điểm số giữa các sinh viên không chênh lệch quá lớn và có xuất hiện đồng hạng.

In [ ]:

show("3b. Xếp hạng theo điểm số trong từng lớp", '''
SELECT student_id, name, class, course_id, score,
       DENSE_RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
ORDER BY class, rank_in_class, student_id;
''')

show("Top 3 cao nhất trong từng lớp", '''
WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY class, rn;
''')

show("Top 3 thấp nhất trong từng lớp", '''
WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score ASC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY class, rn;
''')


### 3b. Xếp hạng theo điểm số trong từng lớp

SELECT student_id, name, class, course_id, score,
       DENSE_RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
ORDER BY class, rank_in_class, student_id;


,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,2
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


### Top 3 cao nhất trong từng lớp

WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY class, rn;


,student_id,name,class,course_id,score,rn
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


### Top 3 thấp nhất trong từng lớp

WITH ranked AS (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score ASC, student_id) AS rn
    FROM student
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY class, rn;


,student_id,name,class,course_id,score,rn
0,9,Duong Huu Phuc,Kinh Te,34,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,3
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,12,7.0,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Kết luận: Kết quả xếp hạng cho thấy có sự đồng hạng ở vị trí cao nhất trong lớp Kinh Tế, trong khi các lớp khác có ít sinh viên hơn nên số hạng đơn giản.
Nhìn chung, điểm số giữa các sinh viên không chênh lệch nhiều và phân bố tương đối đồng đều giữa các lớp.

In [ ]:

show("3c. Xếp hạng theo điểm số trong từng mã môn học", '''
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
       DENSE_RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
FROM student s
JOIN course c
    ON s.course_id = c.id
ORDER BY s.course_id, rank_in_course, s.student_id;
''')

show("Top 3 cao nhất trong từng mã môn học", '''
WITH ranked AS (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           ROW_NUMBER() OVER (PARTITION BY s.course_id ORDER BY s.score DESC, s.student_id) AS rn
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY course_id, rn;
''')

show("Top 3 thấp nhất trong từng mã môn học", '''
WITH ranked AS (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           ROW_NUMBER() OVER (PARTITION BY s.course_id ORDER BY s.score ASC, s.student_id) AS rn
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY course_id, rn;
''')


### 3c. Xếp hạng theo điểm số trong từng mã môn học

SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
       DENSE_RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
FROM student s
JOIN course c
    ON s.course_id = c.id
ORDER BY s.course_id, rank_in_course, s.student_id;


,student_id,name,class,course_id,course_name,score,rank_in_course
0,10,Cao Thi Hanh,May Tinh,12,Giai tich,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,3,Pham Van Nam,Toan Tin,26,Tin hoc,7.9,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,2


### Top 3 cao nhất trong từng mã môn học

WITH ranked AS (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           ROW_NUMBER() OVER (PARTITION BY s.course_id ORDER BY s.score DESC, s.student_id) AS rn
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY course_id, rn;


,student_id,name,class,course_id,course_name,score,rn
0,10,Cao Thi Hanh,May Tinh,12,Giai tich,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,3,Pham Van Nam,Toan Tin,26,Tin hoc,7.9,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,2
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,3


### Top 3 thấp nhất trong từng mã môn học

WITH ranked AS (
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score,
           ROW_NUMBER() OVER (PARTITION BY s.course_id ORDER BY s.score ASC, s.student_id) AS rn
    FROM student s
    JOIN course c
        ON s.course_id = c.id
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY course_id, rn;


,student_id,name,class,course_id,course_name,score,rn
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,Giai tich,7.0,2
2,3,Pham Van Nam,Toan Tin,26,Tin hoc,7.9,1
3,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,3


Kết luận: Kết quả cho thấy mỗi môn học đều có sinh viên đứng đầu, trong đó môn Thống kê có sự đồng hạng cao nhất, còn các môn khác có ít sinh viên nên xếp hạng đơn giản hơn.

## 4.Bổ sung trường `graduation_date`

In [ ]:

alter_update_sql = '''
ALTER TABLE student ADD COLUMN graduation_date DATETIME;

WITH ranked AS (
    SELECT student_id,
           DENSE_RANK() OVER (ORDER BY score DESC) AS rank_by_score
    FROM student
)
UPDATE student
SET graduation_date = datetime(
    'now',
    '+' || (
        SELECT rank_by_score
        FROM ranked
        WHERE ranked.student_id = student.student_id
    ) || ' days'
);
'''

conn.executescript(alter_update_sql)

display(Markdown("### SQL đã dùng"))
print(alter_update_sql.strip())

display(Markdown("### Kết quả bảng `student` sau khi thêm `graduation_date`"))
display(q('''
SELECT student_id, name, class, course_id, score, graduation_date
FROM student
ORDER BY score DESC, student_id;
'''))


### SQL đã dùng

ALTER TABLE student ADD COLUMN graduation_date DATETIME;

WITH ranked AS (
    SELECT student_id,
           DENSE_RANK() OVER (ORDER BY score DESC) AS rank_by_score
    FROM student
)
UPDATE student
SET graduation_date = datetime(
    'now',
    '+' || (
        SELECT rank_by_score
        FROM ranked
        WHERE ranked.student_id = student.student_id
    ) || ' days'
);


### Kết quả bảng `student` sau khi thêm `graduation_date`

,student_id,name,class,course_id,score,graduation_date
0,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-02 15:41:34
1,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-02 15:41:34
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-03 15:41:34
3,9,Duong Huu Phuc,Kinh Te,34,7.2,2026-04-04 15:41:34
4,10,Cao Thi Hanh,May Tinh,12,7.0,2026-04-05 15:41:34
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-06 15:41:34


Kết luận: Trường graduation_date đã được thêm và cập nhật dựa trên thứ hạng điểm số, trong đó sinh viên có điểm cao hơn sẽ có ngày tốt nghiệp sớm hơn. Kết quả đảm bảo dữ liệu hợp lý và phản ánh đúng thứ tự xếp hạng.


## 5.Kết luận ngắn

Sau khi làm sạch dữ liệu theo giả định đã nêu:

- Bảng `student` còn **6 sinh viên hợp lệ**
- Điểm trung bình theo lớp:
  - `Kinh Te`: **8.53**
  - `May Tinh`: **6.85**
  - `Toan Tin`: **7.90**
- Điểm trung bình theo môn:
  - `12 - Giai tich`: **6.85**
  - `26 - Tin hoc`: **7.90**
  - `34 - Thong ke`: **8.53**
- Cả 3 môn đều được xếp loại **Tốt**
- `graduation_date` được tính bằng:
  - **thời điểm chạy notebook + hạng điểm số (DENSE_RANK) tính theo ngày**
